## Local Map Generation using Folium


In [1]:
import folium as flm
import pandas as pd
import json
import os
from glob import glob
import time

In [2]:
def load_latest():
    """Defining a function to load the latest GPS data from the gps_landing directory."""
    latest_file = max(glob("gps_landing/*.json"), key=os.path.getmtime)
    gps_data = pd.read_json(latest_file, lines=True)
    return gps_data


In [3]:
def build_map(gps_data):
    """Building a map with the GPS data, where each point is represented by a circle marker. 
    The color of the marker indicates whether the delay is greater than 120 seconds (red) or not (green)."""
    map = flm.Map(location=[54.3679406,18.5722705], zoom_start=12, title="Gdańsk Map")
    lat = gps_data['lat']
    lon = gps_data['lon']
    n_total = len(gps_data)
    n_delayed = int((gps_data['delay'] > 120).sum())
    
    for i in range(len(gps_data)):
        fill_color = 'red' if gps_data['delay'].iloc[i] > 120 else 'green'
        flm.CircleMarker(
            location=[lat.iloc[i], lon.iloc[i]],
            radius=5,
            fill=True,
            fill_color=fill_color,
            fill_opacity=0.9,
            stroke=False,
            popup=f"{gps_data['routeShortName'].iloc[i]} | Delay: {gps_data['delay'].iloc[i]} s",
            ).add_to(map)
        
    # Automatically refresh the map every 20 seconds    
    map.get_root().header.add_child(
        flm.Element('<meta http-equiv="refresh" content="20">'))
    
    # Keep zoom/pan across the refresh - localStorage
    map_name = map.get_name()
    view_js = f"""
    <script>
    (function() {{
        var KEY = "gdansk_map_view";
        function setup() {{
            var saved = localStorage.getItem(KEY);
            if (saved) {{
                var v = JSON.parse(saved);
                {map_name}.setView([v.lat, v.lng], v.zoom, {{animate: false}});   // restore last view
            }}
            {map_name}.on("moveend", function() {{            // remember on pan/zoom
                var c = {map_name}.getCenter();
                localStorage.setItem(KEY, JSON.stringify(
                    {{lat: c.lat, lng: c.lng, zoom: {map_name}.getZoom()}}));
            }});
        }}
        document.addEventListener("DOMContentLoaded", setup);
    }})();
    </script>
    """
    map.get_root().html.add_child(flm.Element(view_js))
    
    # Legend with total and delayed counts
    legend_html = f"""
    <div style="position: fixed; bottom: 25px; left: 25px; z-index: 1000;
                background: white; padding: 10px 14px; border: 1px solid #999;
                border-radius: 6px; font-family: sans-serif; font-size: 13px;
                box-shadow: 0 1px 4px rgba(0,0,0,.3);">
      <b>Pojazdy: {n_total}</b><br>
      <span style="color:red;">&#9679;</span> Opóźnione &gt; 120 s ({n_delayed})<br>
      <span style="color:green;">&#9679;</span> Na czas ({n_total - n_delayed})
    </div>
    """
    map.get_root().html.add_child(flm.Element(legend_html))
    map.save("map.html")
    
    return map

### Map view persistence (JavaScript)

This snippet keeps the map's zoom and pan position across the 20-second
auto-refresh. Whenever the map is moved, it saves the current center and zoom
level to the browser's `localStorage` (on the `moveend` event). When the page
reloads, it reads those values back and restores the view instantly
(`setView` with `animate: false`), so the refresh no longer snaps the map back
to its default position. The map object is referenced through Folium's
generated name, so nothing needs to be wired up by hand.

_JavaScript generated by Claude._

In [5]:
# Running the map generation in a loop, refreshing every 20 seconds, 
# and allowing for graceful exit on keyboard interrupt.

try:
    for _ in range(20):            # ~6-7 min, similar to the time it takes to run the gps_producer script
        build_map(load_latest())
        time.sleep(20)
except KeyboardInterrupt:
    print("Map generation stopped.")


Map generation stopped.
